# 🤖 AI Agent Builder
### 🛠️ Developed by: **Arafat Ahmed Mubin**
### 🌐 Brand: **2M**
---


### Rapid Deployment Framework for Custom LLM Personas

This notebook provides a complete platform to design, test, and deploy AI agents using the **OpenRouter API**. 

**Key Capabilities:**
- **Free Model Rotation:** Automatically utilizes `openrouter/free` to access top models (Llama 3, Mistral, Gemma) at zero cost.
- **Persona Designer:** Deep configuration of System Prompts and behavioral parameters.
- **Template Library:** Instant deployment of specialized agents for Coding, SEO, Storytelling, and Math.
- **Security:** Encrypted handling of API keys and robust error management.

## 🛠️ Step 1: Environment Setup
Install the OpenAI SDK (configured for OpenRouter) and Gradio for the dashboard.

In [ ]:
# Install core libraries
!pip install -q -U openai gradio python-dotenv

## 🧠 Step 2: Agent Architecture Logic
This section defines the template library and the communication logic with OpenRouter.

In [ ]:
import os
from openai import OpenAI
import gradio as gr
import json
import base64

# Template Library
TEMPLATES = {
    "Professional Coder": "You are a Senior Software Architect. Your mission is to provide clean, efficient, and well-documented code.",
    "SEO & Content Master": "You are a world-class SEO expert and copywriter.",
    "Creative Storyteller": "You are a master novelist and screenwriter.",
    "Math & Logic Analyst": "You are a Mathematical Genius."
}

def get_client(api_key):
    if not api_key: return None
    return OpenAI(base_url="https://openrouter.ai/api/v1", api_key=api_key)

def chat_with_agent(api_key, system_prompt, temperature, message, history):
    client = get_client(api_key)
    if not client: return history + [("Error", "Please enter a valid API Key.")]
    try:
        messages = [{"role": "system", "content": system_prompt}]
        for user_msg, assistant_msg in history:
            messages.append({"role": "user", "content": user_msg})
            messages.append({"role": "assistant", "content": assistant_msg})
        messages.append({"role": "user", "content": message})
        response = client.chat.completions.create(
            model="openrouter/auto",
            messages=messages,
            temperature=temperature
        )
        reply = response.choices[0].message.content
        history.append((message, reply))
        return history, ""
    except Exception as e:
        history.append((message, f"⚠️ API Error: {str(e)}"))
        return history, ""


## 🏗️ Step 3: Agent Builder Interface
Launch the 3-tab platform: Setup, Designer, and Live Chat.

In [ ]:
def generate_share_code(prompt, temp):
    data = {"p": prompt, "t": temp}
    encoded = base64.b64encode(json.dumps(data).encode()).decode()
    return f"AGENT_CFG_{encoded}"

with gr.Blocks(theme=gr.themes.Soft()) as builder:
    gr.Markdown("# 🤖 AI Agent Builder Studio")
    api_key_state = gr.State("")
    with gr.Tabs() as tabs:
        with gr.Tab("🔑 Setup"):
            key_input = gr.Textbox(label="OpenRouter API Key", type="password")
            save_btn = gr.Button("Initialize")
            setup_status = gr.Markdown("Status: 🔴 Not Connected")
            save_btn.click(lambda k: (k, "Status: 🟢 Connected") if k.startswith("sk-or-") else ("", "Status: ⚠️ Error"), key_input, [api_key_state, setup_status])
        with gr.Tab("🎨 Designer"):
            template_drop = gr.Dropdown(list(TEMPLATES.keys()), label="Templates")
            system_prompt = gr.Textbox(label="System Prompt", lines=10, value=TEMPLATES["Professional Coder"])
            temp_slider = gr.Slider(0.0, 2.0, value=0.7, label="Temperature")
            template_drop.change(lambda t: TEMPLATES[t], template_drop, system_prompt)
        with gr.Tab("💬 Live Chat"):
            chatbot = gr.Chatbot(label="Chat")
            msg_input = gr.Textbox(label="Message")
            submit_btn = gr.Button("Send")
            submit_btn.click(chat_with_agent, [api_key_state, system_prompt, temp_slider, msg_input, chatbot], [chatbot, msg_input])

builder.launch(share=True, debug=True)

---
**© 2026 2M | All Rights Reserved**
*Powered by the 2M Ecosystem (2M Dev's, 2M Future Facts, 2M Business Blogs)*